In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [3]:
# ========================================================================
# 1. LOAD DATA AND CREATE REVENUE PROXY
# ========================================================================

print("=" * 70)
print("GAME AGE ANALYSIS - REVENUE PROXY INVESTIGATION")
print("=" * 70)

# Load cleaned dataset
df = pd.read_csv('steam_games_cleaned.csv')

print(f"\n✓ Dataset loaded: {df.shape[0]:,} games")

# Create revenue proxy
REVIEW_TO_OWNER_RATIO = 75
F2P_REVENUE_PER_PLAYER = 50

df['estimated_owners'] = df['overall_review_count'] * REVIEW_TO_OWNER_RATIO
df['estimated_revenue'] = df['estimated_owners'] * df['final_price']
df.loc[df['is_free_to_play'] == True, 'estimated_revenue'] = (
    df.loc[df['is_free_to_play'] == True, 'estimated_owners'] * F2P_REVENUE_PER_PLAYER
)

# Filter to games with complete data
df_complete = df[
    (df['game_age_years'].notna()) &
    (df['estimated_revenue'].notna()) &
    (df['estimated_revenue'] > 0)
].copy()

print(f"Games with complete age & revenue data: {len(df_complete):,}")

GAME AGE ANALYSIS - REVENUE PROXY INVESTIGATION

✓ Dataset loaded: 42,497 games
Games with complete age & revenue data: 39,722


In [4]:
# ========================================================================
# 2. CORRELATION ANALYSIS
# ========================================================================

print("\n" + "=" * 70)
print("APPROACH 1: CORRELATION ANALYSIS")
print("=" * 70)

# Calculate correlations
correlations = df_complete[
    ['game_age_years', 'estimated_revenue', 'overall_review_count',
     'final_price', 'overall_review_pct']
].corr()

print("\nCorrelation Matrix (focusing on game_age_years):")
print(correlations['game_age_years'].sort_values(ascending=False))

# Interpretation
age_revenue_corr = correlations.loc['game_age_years', 'estimated_revenue']
print(f"\n📊 Key Finding:")
print(f"   Game Age vs Revenue correlation: {age_revenue_corr:.3f}")

if abs(age_revenue_corr) < 0.1:
    print("   → WEAK correlation - age alone is not a strong revenue predictor")
elif abs(age_revenue_corr) < 0.3:
    print("   → MODERATE correlation - age has some influence on revenue")
else:
    print("   → STRONG correlation - age significantly affects revenue")


APPROACH 1: CORRELATION ANALYSIS

Correlation Matrix (focusing on game_age_years):
game_age_years          1.000000
overall_review_count    0.038101
estimated_revenue       0.014422
final_price            -0.066787
overall_review_pct     -0.169916
Name: game_age_years, dtype: float64

📊 Key Finding:
   Game Age vs Revenue correlation: 0.014
   → WEAK correlation - age alone is not a strong revenue predictor


In [5]:
# ========================================================================
# 3. COHORT ANALYSIS BY AGE BUCKET
# ========================================================================

print("\n" + "=" * 70)
print("APPROACH 2: COHORT ANALYSIS BY AGE BUCKET")
print("=" * 70)

# Group by age bucket
age_cohort = df_complete.groupby('age_bucket').agg({
    'estimated_revenue': ['count', 'median', 'mean', 'sum'],
    'overall_review_count': 'median',
    'final_price': 'median',
    'overall_review_pct': 'median'
}).round(2)

age_cohort.columns = [
    'game_count', 'median_revenue', 'mean_revenue', 'total_revenue',
    'median_reviews', 'median_price', 'median_rating'
]

# Convert to millions/billions
age_cohort['median_revenue_M'] = (age_cohort['median_revenue'] / 1e6).round(2)
age_cohort['mean_revenue_M'] = (age_cohort['mean_revenue'] / 1e6).round(2)
age_cohort['total_revenue_B'] = (age_cohort['total_revenue'] / 1e9).round(2)

print("\nRevenue Patterns by Game Age:\n")
print(age_cohort[[
    'game_count', 'median_revenue_M', 'mean_revenue_M',
    'total_revenue_B', 'median_reviews', 'median_price'
]].to_string())

# Calculate revenue per game ratio (vs newest games)
newest_revenue_per_game = age_cohort.loc['<1y', 'mean_revenue']
age_cohort['revenue_multiplier_vs_new'] = (
    age_cohort['mean_revenue'] / newest_revenue_per_game
).round(2)

print("\n📊 Revenue Multiplier vs New Games (<1y):")
print(age_cohort['revenue_multiplier_vs_new'].to_string())


APPROACH 2: COHORT ANALYSIS BY AGE BUCKET

Revenue Patterns by Game Age:

            game_count  median_revenue_M  mean_revenue_M  total_revenue_B  median_reviews  median_price
age_bucket                                                                                             
1-3y             10312              0.73          116.94          1205.89            41.0         251.5
10y+              2915              7.78          254.21           741.02           369.0         349.0
3-5y              8621              0.65          149.14          1285.74            47.0         200.0
5-10y            15510              0.94          154.86          2401.82            67.0         250.0
<1y               2364              0.81          150.60           356.01            35.0         309.5

📊 Revenue Multiplier vs New Games (<1y):
age_bucket
1-3y     0.78
10y+     1.69
3-5y     0.99
5-10y    1.03
<1y      1.00


In [6]:
# ========================================================================
# 4. SURVIVAL/LONGEVITY ANALYSIS
# ========================================================================

print("\n" + "=" * 70)
print("APPROACH 3: REVENUE LONGEVITY ANALYSIS")
print("=" * 70)

# Analyze what % of revenue comes from each age group
age_cohort['market_share_pct'] = (
    age_cohort['total_revenue'] / age_cohort['total_revenue'].sum() * 100
).round(2)

print("\nMarket Share by Game Age:")
print(age_cohort[['game_count', 'total_revenue_B', 'market_share_pct']].to_string())

# Key insight
oldest_share = age_cohort.loc['10y+', 'market_share_pct']
oldest_count_pct = (
    age_cohort.loc['10y+', 'game_count'] / age_cohort['game_count'].sum() * 100
)

print(f"\n📊 Long-Tail Insight:")
print(f"   Games 10+ years old:")
print(f"   • Represent {oldest_count_pct:.1f}% of game count")
print(f"   • Generate {oldest_share:.1f}% of total revenue")
print(f"   • Revenue per game is {age_cohort.loc['10y+', 'revenue_multiplier_vs_new']:.1f}x higher than new releases")


APPROACH 3: REVENUE LONGEVITY ANALYSIS

Market Share by Game Age:
            game_count  total_revenue_B  market_share_pct
age_bucket                                               
1-3y             10312          1205.89             20.13
10y+              2915           741.02             12.37
3-5y              8621          1285.74             21.46
5-10y            15510          2401.82             40.09
<1y               2364           356.01              5.94

📊 Long-Tail Insight:
   Games 10+ years old:
   • Represent 7.3% of game count
   • Generate 12.4% of total revenue
   • Revenue per game is 1.7x higher than new releases


In [7]:
# ========================================================================
# 5. GENRE × AGE INTERACTION ANALYSIS
# ========================================================================

print("\n" + "=" * 70)
print("APPROACH 4: GENRE × AGE INTERACTION")
print("=" * 70)

# Analyze if certain genres age better than others
top_genres = ['action', 'adventure', 'rpg', 'strategy', 'simulation', 'indie']

genre_age_interaction = df_complete[
    df_complete['main_genre'].isin(top_genres)
].groupby(['main_genre', 'age_bucket']).agg({
    'estimated_revenue': ['count', 'median']
}).round(0)

genre_age_interaction.columns = ['game_count', 'median_revenue']
genre_age_interaction['median_revenue_M'] = (
    genre_age_interaction['median_revenue'] / 1e6
).round(1)

print("\nMedian Revenue (₹M) by Genre × Age:\n")
pivot_table = genre_age_interaction.reset_index().pivot(
    index='main_genre',
    columns='age_bucket',
    values='median_revenue_M'
)

# Reorder columns chronologically
col_order = ['<1y', '1-3y', '3-5y', '5-10y', '10y+']
pivot_table = pivot_table[[col for col in col_order if col in pivot_table.columns]]

print(pivot_table.to_string())

print("\n📊 Genre Aging Patterns:")
for genre in top_genres:
    if genre in pivot_table.index:
        oldest = pivot_table.loc[genre, '10y+'] if '10y+' in pivot_table.columns else np.nan
        newest = pivot_table.loc[genre, '<1y'] if '<1y' in pivot_table.columns else np.nan
        if pd.notna(oldest) and pd.notna(newest) and newest > 0:
            aging_factor = oldest / newest
            print(f"   {genre.capitalize()}: Old games make {aging_factor:.1f}x newer games")


APPROACH 4: GENRE × AGE INTERACTION

Median Revenue (₹M) by Genre × Age:

age_bucket  <1y  1-3y  3-5y  5-10y  10y+
main_genre                              
action      0.9   0.8   0.7    0.9  14.3
adventure   0.9   0.8   0.8    1.1   5.3
indie       0.9   1.0   1.0    1.3   6.2
rpg         4.9   3.5   8.0    9.5  16.8
simulation  0.9   3.4   2.1    2.7   6.9
strategy    1.7   1.2   1.3    3.1  10.9

📊 Genre Aging Patterns:
   Action: Old games make 15.9x newer games
   Adventure: Old games make 5.9x newer games
   Rpg: Old games make 3.4x newer games
   Strategy: Old games make 6.4x newer games
   Simulation: Old games make 7.7x newer games
   Indie: Old games make 6.9x newer games


In [8]:
# ========================================================================
# 6. STATISTICAL SIGNIFICANCE TEST
# ========================================================================

print("\n" + "=" * 70)
print("APPROACH 5: STATISTICAL SIGNIFICANCE TEST")
print("=" * 70)

# ANOVA test: Does revenue significantly differ across age groups?
age_groups = []
for bucket in df_complete['age_bucket'].unique():
    if pd.notna(bucket):
        group_data = df_complete[df_complete['age_bucket'] == bucket]['estimated_revenue']
        # Use log to normalize (revenue is highly skewed)
        age_groups.append(np.log1p(group_data.dropna()))

f_stat, p_value = stats.f_oneway(*age_groups)

print(f"\nANOVA Test: Does revenue differ significantly across age groups?")
print(f"   F-statistic: {f_stat:.2f}")
print(f"   P-value: {p_value:.4f}")

if p_value < 0.01:
    print(f"   → YES, highly significant (p < 0.01)")
    print(f"   → Age DOES matter for revenue prediction")
else:
    print(f"   → NO, not significant (p >= 0.01)")
    print(f"   → Age may NOT be critical for revenue proxy")


APPROACH 5: STATISTICAL SIGNIFICANCE TEST

ANOVA Test: Does revenue differ significantly across age groups?
   F-statistic: 522.48
   P-value: 0.0000
   → YES, highly significant (p < 0.01)
   → Age DOES matter for revenue prediction


In [9]:
# ========================================================================
# 7. REVENUE PROXY RECOMMENDATION
# ========================================================================

print("\n" + "=" * 70)
print("RECOMMENDATION: SHOULD GAME AGE BE IN REVENUE PROXY?")
print("=" * 70)

recommendation = f"""

ANALYSIS SUMMARY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Direct Correlation: {age_revenue_corr:.3f}
   {'✅ Meaningful relationship exists' if abs(age_revenue_corr) > 0.1 else '❌ Very weak direct relationship'}

2. Cohort Performance:
   • Older games (10y+) generate {age_cohort.loc['10y+', 'revenue_multiplier_vs_new']:.1f}x revenue per game vs new
   • BUT this is because survivors are the "winners"

3. Market Share:
   • Oldest 10% of games generate {oldest_share:.1f}% of revenue
   {'✅ Long-tail effect is significant' if oldest_share > 15 else '⚠️ Newer games dominate'}

4. Statistical Test (ANOVA):
   • P-value: {p_value:.4f}
   {'✅ Age groups differ significantly' if p_value < 0.01 else '⚠️ Differences may be random'}

RECOMMENDATION FOR REVENUE PROXY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

{"✅ YES - Include game_age_years as a feature" if p_value < 0.01 and abs(age_revenue_corr) > 0.1 else "⚠️ MAYBE - Use with caution"}

Reasoning:
• The relationship is NOT linear (older ≠ always higher revenue)
• Survivorship bias: Old games on Steam are "winners" that survived
• Better approach: Use age as a FEATURE in ML model, not in base proxy
• Let the ML model learn the complex age-revenue relationship

PROPOSED REVENUE PROXY (Keep Simple):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Current: Revenue = Estimated_Owners × Price
         (Estimated_Owners = Reviews × 75)

✅ KEEP this simple proxy for baseline analysis

For ML Model: Include these features:
• game_age_years (let model learn the pattern)
• age_bucket (categorical, captures non-linear effects)
• genre × age interaction terms
• has_dlc (DLC extends lifetime)

WHY NOT include age in base proxy:
• Survivorship bias would overestimate new game revenue
• Relationship is complex (U-shaped, genre-dependent)
• Better handled by ML feature engineering

"""

print(recommendation)


RECOMMENDATION: SHOULD GAME AGE BE IN REVENUE PROXY?


ANALYSIS SUMMARY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Direct Correlation: 0.014
   ❌ Very weak direct relationship

2. Cohort Performance:
   • Older games (10y+) generate 1.7x revenue per game vs new
   • BUT this is because survivors are the "winners"
   
3. Market Share:
   • Oldest 10% of games generate 12.4% of revenue
   ⚠️ Newer games dominate

4. Statistical Test (ANOVA):
   • P-value: 0.0000
   ✅ Age groups differ significantly

RECOMMENDATION FOR REVENUE PROXY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚠️ MAYBE - Use with caution

Reasoning:
• The relationship is NOT linear (older ≠ always higher revenue)
• Survivorship bias: Old games on Steam are "winners" that survived
• Better approach: Use age as a FEATURE in ML model, not in base proxy
• Let the ML model learn the complex age-revenue relationship

PROPOSED REVENUE PROXY (Keep Simple):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Current: Revenue = Estimated_Owners 